In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

# =====================================================
# Enhanced Class-Conditioned Latent Diffusion Autoencoder
# Potato Leaf Disease Representation Learning
# =====================================================

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder
from torchvision import transforms as T
from torchvision.utils import save_image
from torch.amp import autocast, GradScaler

# =====================================================
# CONFIG
# =====================================================
DATASET_PATH = "/content/drive/MyDrive/POTATODATASET/PLDdatasetLDM"
SAVE_DIR = "/content/drive/MyDrive/POTATODATASET/Potato_LDMClassifierTrain_Output"

IMG_SIZE = 256
LATENT_DIM = 512
NUM_CLASSES = 3

BATCH_SIZE = 16
EPOCHS = 100

LR = 1e-4
TIMESTEPS = 1000

os.makedirs(SAVE_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scaler = GradScaler(device="cuda" if torch.cuda.is_available() else "cpu")

# =====================================================
# DATASET
# =====================================================
transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])

dataset = ImageFolder(DATASET_PATH, transform=transform)

print("="*60)
print("Classes:", dataset.classes)
print("Class Mapping:", dataset.class_to_idx)
print("Total Images:", len(dataset))
print("="*60)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# =====================================================
# MODEL BLOCKS
# =====================================================
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, 1, 1)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.skip = nn.Identity() if in_ch == out_ch else nn.Conv2d(in_ch, out_ch, 1)

    def forward(self, x):
        identity = self.skip(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + identity)

class SelfAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.query = nn.Conv2d(channels, channels // 8, 1)
        self.key = nn.Conv2d(channels, channels // 8, 1)
        self.value = nn.Conv2d(channels, channels, 1)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        b, c, h, w = x.size()
        q = self.query(x).view(b, -1, h*w).permute(0, 2, 1)
        k = self.key(x).view(b, -1, h*w)
        attn = torch.softmax(torch.bmm(q, k), dim=-1)
        v = self.value(x).view(b, c, h*w)
        out = torch.bmm(v, attn.permute(0, 2, 1)).view(b, c, h, w)
        return self.gamma * out + x

class Encoder(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.ReLU(),
            ResidualBlock(64, 64),
            nn.Conv2d(64, 128, 4, 2, 1), ResidualBlock(128, 128),
            nn.Conv2d(128, 256, 4, 2, 1), ResidualBlock(256, 256),
            nn.Conv2d(256, 512, 4, 2, 1), ResidualBlock(512, 512),
            SelfAttention(512),
            nn.Conv2d(512, 512, 4, 2, 1), nn.ReLU()
        )
        self.fc = nn.Linear(512 * 8 * 8, latent_dim)

    def forward(self, x):
        return self.fc(torch.flatten(self.features(x), 1))

class Decoder(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 512 * 8 * 8)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(512, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Tanh()
        )

    def forward(self, z):
        x = self.fc(z).view(-1, 512, 8, 8)
        return self.net(x)

class LatentDenoiser(nn.Module):
    def __init__(self, latent_dim, timesteps, num_classes):
        super().__init__()
        self.time_embed = nn.Embedding(timesteps, latent_dim)
        self.class_embed = nn.Embedding(num_classes, latent_dim)
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 1024), nn.ReLU(),
            nn.Linear(1024, 1024), nn.ReLU(),
            nn.Linear(1024, latent_dim)
        )

    def forward(self, z, t, y):
        cond = self.time_embed(t) + self.class_embed(y)
        return self.net(z + cond)

encoder = Encoder(LATENT_DIM).to(device)
decoder = Decoder(LATENT_DIM).to(device)
denoiser = LatentDenoiser(LATENT_DIM, TIMESTEPS, NUM_CLASSES).to(device)

optimizer = optim.Adam(
    list(encoder.parameters()) +
    list(decoder.parameters()) +
    list(denoiser.parameters()),
    lr=LR
)

betas = torch.linspace(1e-4, 0.02, TIMESTEPS).to(device)
alphas = 1.0 - betas
alpha_hat = torch.cumprod(alphas, dim=0)

def kl_loss(z):
    return 0.5 * torch.mean(z.pow(2))

best_val = 1e9
patience = 20
counter = 0

for epoch in range(EPOCHS):

    encoder.train(); decoder.train(); denoiser.train()
    train_loss = 0

    for imgs, labels in train_loader:

        imgs = imgs.to(device)
        labels = labels.to(device)

        with autocast(device_type=device.type):
            z = encoder(imgs)
            recon = decoder(z)

            recon_loss = F.mse_loss(recon, imgs)
            kll = kl_loss(z)

            t = torch.randint(0, TIMESTEPS, (z.size(0),), device=device)
            noise = torch.randn_like(z)

            ah = alpha_hat[t].unsqueeze(1)
            noisy_z = torch.sqrt(ah) * z + torch.sqrt(1 - ah) * noise

            pred_noise = denoiser(noisy_z, t, labels)
            diffusion_loss = F.mse_loss(pred_noise, noise)

            total_loss = recon_loss + diffusion_loss + 0.001 * kll

        optimizer.zero_grad()
        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += total_loss.item()

    train_loss /= len(train_loader)

    encoder.eval(); decoder.eval(); denoiser.eval()
    val_loss = 0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            z = encoder(imgs)
            recon = decoder(z)

            recon_loss = F.mse_loss(recon, imgs)

            t = torch.randint(0, TIMESTEPS, (z.size(0),), device=device)
            noise = torch.randn_like(z)

            ah = alpha_hat[t].unsqueeze(1)
            noisy_z = torch.sqrt(ah) * z + torch.sqrt(1 - ah) * noise

            pred_noise = denoiser(noisy_z, t, labels)
            diffusion_loss = F.mse_loss(pred_noise, noise)

            val_loss += (recon_loss + diffusion_loss).item()

    val_loss /= len(val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train {train_loss:.4f} | Val {val_loss:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        counter = 0

        torch.save(encoder.state_dict(), os.path.join(SAVE_DIR, "encoder_best.pth"))
        torch.save(decoder.state_dict(), os.path.join(SAVE_DIR, "decoder_best.pth"))
        torch.save(denoiser.state_dict(), os.path.join(SAVE_DIR, "denoiser_best.pth"))

        torch.save({
            "epoch": epoch,
            "encoder": encoder.state_dict(),
            "decoder": decoder.state_dict(),
            "denoiser": denoiser.state_dict(),
            "optimizer": optimizer.state_dict(),
            "val_loss": val_loss
        }, os.path.join(SAVE_DIR, "best_checkpoint.pth"))

    else:
        counter += 1

    if counter >= patience:
        print("Early stopping")
        break

    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            sample_imgs, _ = next(iter(val_loader))
            sample_imgs = sample_imgs[:8].to(device)
            recon = decoder(encoder(sample_imgs))

            save_image(
                (recon + 1) / 2,
                os.path.join(SAVE_DIR, f"epoch_{epoch+1}.png"),
                nrow=4
            )

torch.save(
    encoder.state_dict(),
    os.path.join(SAVE_DIR, "encoder_final.pth")
)

print("Training completed")


Classes: ['EarlyBlight', 'Healthy', 'LateBlight']
Class Mapping: {'EarlyBlight': 0, 'Healthy': 1, 'LateBlight': 2}
Total Images: 4072
Epoch 1/100 | Train 1.1316 | Val 1.0468
Epoch 2/100 | Train 1.0303 | Val 1.0167
Epoch 3/100 | Train 0.9944 | Val 0.9721
Epoch 4/100 | Train 0.9465 | Val 0.9253
Epoch 5/100 | Train 0.8947 | Val 0.8745
Epoch 6/100 | Train 0.8540 | Val 0.8445
Epoch 7/100 | Train 0.8223 | Val 0.8091
Epoch 8/100 | Train 0.7995 | Val 0.7845
Epoch 9/100 | Train 0.7809 | Val 0.7739
Epoch 10/100 | Train 0.7590 | Val 0.7507
Epoch 11/100 | Train 0.7439 | Val 0.7437
Epoch 12/100 | Train 0.7251 | Val 0.7213
Epoch 13/100 | Train 0.7158 | Val 0.7122
Epoch 14/100 | Train 0.7026 | Val 0.6977
Epoch 15/100 | Train 0.6961 | Val 0.6910
Epoch 16/100 | Train 0.6833 | Val 0.6897
Epoch 17/100 | Train 0.6726 | Val 0.6666
Epoch 18/100 | Train 0.6687 | Val 0.6738
Epoch 19/100 | Train 0.6560 | Val 0.6494
Epoch 20/100 | Train 0.6517 | Val 0.6473
Epoch 21/100 | Train 0.6455 | Val 0.6403
Epoch 22/100 |

In [ ]:

# =====================================================
# Enhanced Class-Conditioned Latent Diffusion Autoencoder
# Potato Leaf Disease Representation Learning
# =====================================================

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder
from torchvision import transforms as T
from torchvision.utils import save_image
from torch.amp import autocast, GradScaler

# =====================================================
# CONFIG
# =====================================================
DATASET_PATH = "/content/drive/MyDrive/POTATODATASET/PLDdatasetLDM"
SAVE_DIR = "/content/drive/MyDrive/POTATODATASET/Potato_LDMClassifierTrainEnhnace_Output"

IMG_SIZE = 256
LATENT_DIM = 1024
NUM_CLASSES = 3

BATCH_SIZE = 16
EPOCHS = 100

LR = 1e-4
TIMESTEPS = 1000

os.makedirs(SAVE_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scaler = GradScaler(device="cuda" if torch.cuda.is_available() else "cpu")

# =====================================================
# DATASET
# =====================================================
transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])

dataset = ImageFolder(DATASET_PATH, transform=transform)

print("="*60)
print("Classes:", dataset.classes)
print("Class Mapping:", dataset.class_to_idx)
print("Total Images:", len(dataset))
print("="*60)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# =====================================================
# MODEL BLOCKS
# =====================================================
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, 1, 1)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.skip = nn.Identity() if in_ch == out_ch else nn.Conv2d(in_ch, out_ch, 1)

    def forward(self, x):
        identity = self.skip(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + identity)

class SelfAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.query = nn.Conv2d(channels, channels // 8, 1)
        self.key = nn.Conv2d(channels, channels // 8, 1)
        self.value = nn.Conv2d(channels, channels, 1)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        b, c, h, w = x.size()
        q = self.query(x).view(b, -1, h*w).permute(0, 2, 1)
        k = self.key(x).view(b, -1, h*w)
        attn = torch.softmax(torch.bmm(q, k), dim=-1)
        v = self.value(x).view(b, c, h*w)
        out = torch.bmm(v, attn.permute(0, 2, 1)).view(b, c, h, w)
        return self.gamma * out + x

class Encoder(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.ReLU(),
            ResidualBlock(64, 64),
            nn.Conv2d(64, 128, 4, 2, 1), ResidualBlock(128, 128),
            nn.Conv2d(128, 256, 4, 2, 1), ResidualBlock(256, 256),
            nn.Conv2d(256, 512, 4, 2, 1), ResidualBlock(512, 512),
            SelfAttention(512),
            nn.Conv2d(512, 512, 4, 2, 1), nn.ReLU()
        )
        self.fc = nn.Linear(512 * 8 * 8, latent_dim)

    def forward(self, x):
        return self.fc(torch.flatten(self.features(x), 1))

class Decoder(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 512 * 8 * 8)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(512, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Tanh()
        )

    def forward(self, z):
        x = self.fc(z).view(-1, 512, 8, 8)
        return self.net(x)

class LatentDenoiser(nn.Module):
    def __init__(self, latent_dim, timesteps, num_classes):
        super().__init__()
        self.time_embed = nn.Embedding(timesteps, latent_dim)
        self.class_embed = nn.Embedding(num_classes, latent_dim)
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 1024), nn.ReLU(),
            nn.Linear(1024, 1024), nn.ReLU(),
            nn.Linear(1024, latent_dim)
        )

    def forward(self, z, t, y):
        cond = self.time_embed(t) + self.class_embed(y)
        return self.net(z + cond)

        # =====================================================
# CLASSIFICATION HEAD
# =====================================================

class ClassifierHead(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(LATENT_DIM, 512),

            nn.BatchNorm1d(512),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(512, 256),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(256, NUM_CLASSES)

        )

    def forward(self, z):

        return self.net(z)

encoder = Encoder(LATENT_DIM).to(device)

decoder = Decoder(LATENT_DIM).to(device)

denoiser = LatentDenoiser(
    LATENT_DIM,
    TIMESTEPS,
    NUM_CLASSES
).to(device)

classifier = ClassifierHead().to(device)

optimizer = optim.AdamW(

    list(encoder.parameters()) +

    list(decoder.parameters()) +

    list(denoiser.parameters()) +

    list(classifier.parameters()),

    lr=LR,

    weight_decay=1e-4

)

betas = torch.linspace(1e-4, 0.02, TIMESTEPS).to(device)
alphas = 1.0 - betas
alpha_hat = torch.cumprod(alphas, dim=0)

def kl_loss(z):
    return 0.5 * torch.mean(z.pow(2))

best_val = 1e9
patience = 20
counter = 0

for epoch in range(EPOCHS):

    encoder.train()
    decoder.train()
    denoiser.train()
    classifier.train()

    train_loss = 0

    for imgs, labels in train_loader:

        imgs = imgs.to(device)
        labels = labels.to(device)

        with autocast(device_type=device.type):

            # -------------------------
            # Encode
            # -------------------------
            z = encoder(imgs)

            # -------------------------
            # Classification Loss
            # -------------------------
            logits = classifier(z)
            cls_loss = F.cross_entropy(logits, labels)

            # -------------------------
            # Reconstruction Loss
            # -------------------------
            recon = decoder(z)
            recon_loss = F.l1_loss(recon, imgs)

            # -------------------------
            # KL Loss
            # -------------------------
            kll = kl_loss(z)

            # -------------------------
            # Diffusion Loss
            # -------------------------
            t = torch.randint(
                0,
                TIMESTEPS,
                (z.size(0),),
                device=device
            )

            noise = torch.randn_like(z)

            ah = alpha_hat[t].unsqueeze(1)

            noisy_z = (
                torch.sqrt(ah) * z +
                torch.sqrt(1 - ah) * noise
            )

            pred_noise = denoiser(
                noisy_z,
                t,
                labels
            )

            diffusion_loss = F.mse_loss(
                pred_noise,
                noise
            )

            total_loss = (
                recon_loss +
                diffusion_loss +
                0.5 * cls_loss +
                0.001 * kll
            )

        optimizer.zero_grad()

        scaler.scale(total_loss).backward()

        scaler.step(optimizer)

        scaler.update()

        train_loss += total_loss.item()

    train_loss /= len(train_loader)

    # ==================================================
    # Validation
    # ==================================================
    encoder.eval()
    decoder.eval()
    denoiser.eval()
    classifier.eval()

    val_loss = 0

    with torch.no_grad():

        for imgs, labels in val_loader:

            imgs = imgs.to(device)
            labels = labels.to(device)

            z = encoder(imgs)

            # Classification
            logits = classifier(z)
            cls_loss = F.cross_entropy(
                logits,
                labels
            )

            # Reconstruction
            recon = decoder(z)

            recon_loss = F.l1_loss(
                recon,
                imgs
            )

            # Diffusion
            t = torch.randint(
                0,
                TIMESTEPS,
                (z.size(0),),
                device=device
            )

            noise = torch.randn_like(z)

            ah = alpha_hat[t].unsqueeze(1)

            noisy_z = (
                torch.sqrt(ah) * z +
                torch.sqrt(1 - ah) * noise
            )

            pred_noise = denoiser(
                noisy_z,
                t,
                labels
            )

            diffusion_loss = F.mse_loss(
                pred_noise,
                noise
            )

            val_loss += (
                recon_loss +
                diffusion_loss +
                0.5 * cls_loss
            ).item()

    val_loss /= len(val_loader)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train {train_loss:.4f} | "
        f"Val {val_loss:.4f}"
    )

    # ==================================================
    # Save Best Model
    # ==================================================
    if val_loss < best_val:

        best_val = val_loss
        counter = 0

        torch.save(
            encoder.state_dict(),
            os.path.join(
                SAVE_DIR,
                "encoder_best.pth"
            )
        )

        torch.save(
            decoder.state_dict(),
            os.path.join(
                SAVE_DIR,
                "decoder_best.pth"
            )
        )

        torch.save(
            denoiser.state_dict(),
            os.path.join(
                SAVE_DIR,
                "denoiser_best.pth"
            )
        )

        torch.save(
            classifier.state_dict(),
            os.path.join(
                SAVE_DIR,
                "classifier_best.pth"
            )
        )

        torch.save(
            {
                "epoch": epoch,
                "encoder": encoder.state_dict(),
                "decoder": decoder.state_dict(),
                "denoiser": denoiser.state_dict(),
                "classifier": classifier.state_dict(),
                "optimizer": optimizer.state_dict(),
                "val_loss": val_loss
            },
            os.path.join(
                SAVE_DIR,
                "checkpoint_best.pth"
            )
        )

    else:
        counter += 1

    # ==================================================
    # Early Stopping
    # ==================================================
    if counter >= patience:
        print("Early stopping")
        break

    # ==================================================
    # Save Reconstructions
    # ==================================================
    if (epoch + 1) % 10 == 0:

        with torch.no_grad():

            sample_imgs, _ = next(iter(val_loader))

            sample_imgs = sample_imgs[:8].to(device)

            recon = decoder(
                encoder(sample_imgs)
            )

            save_image(
                (recon + 1) / 2,
                os.path.join(
                    SAVE_DIR,
                    f"epoch_{epoch+1}.png"
                ),
                nrow=4
            )

# ==================================================
# Final Classifier Save
# ==================================================
torch.save(
    classifier.state_dict(),
    os.path.join(
        SAVE_DIR,
        "classifier_final.pth"
    )
)

print("Training completed")


Classes: ['EarlyBlight', 'Healthy', 'LateBlight']
Class Mapping: {'EarlyBlight': 0, 'Healthy': 1, 'LateBlight': 2}
Total Images: 4072
Epoch 1/100 | Train 1.5785 | Val 1.4535
Epoch 2/100 | Train 1.3845 | Val 1.3340
Epoch 3/100 | Train 1.3120 | Val 1.2292
Epoch 4/100 | Train 1.2529 | Val 1.2515
Epoch 5/100 | Train 1.2207 | Val 1.2516
Epoch 6/100 | Train 1.2092 | Val 1.2094
Epoch 7/100 | Train 1.1808 | Val 1.1788
Epoch 8/100 | Train 1.1509 | Val 1.1080
Epoch 9/100 | Train 1.1139 | Val 1.1092
Epoch 10/100 | Train 1.1016 | Val 1.1381
Epoch 11/100 | Train 1.0820 | Val 1.1078
Epoch 12/100 | Train 1.0679 | Val 1.0545
Epoch 13/100 | Train 1.0518 | Val 1.0360
Epoch 14/100 | Train 1.0305 | Val 1.0341
Epoch 15/100 | Train 1.0096 | Val 1.0219
Epoch 16/100 | Train 1.0127 | Val 1.0528
Epoch 17/100 | Train 1.0045 | Val 1.0204
Epoch 18/100 | Train 0.9813 | Val 0.9958
Epoch 19/100 | Train 0.9873 | Val 1.0427
Epoch 20/100 | Train 0.9907 | Val 0.9881
Epoch 21/100 | Train 0.9573 | Val 1.0075
Epoch 22/100 |